# Federated Learning — FedOpt on Cotton Plant Disease Dataset

**Algorithm:** FedOpt (server-side Adam over aggregated client deltas)
**Model:** ResNet18 (trained from scratch, no pretrained weights — keeps this notebook fully offline / no internet needed)
**Clients:** 5, Dirichlet non-IID partition (α = 0.5)
**Local training:** SGD (lr=0.001, momentum=0.9), batch size 32, 5 local epochs
**Global rounds:** 10
**Server optimizer:** Adam, lr=0.001

**Dataset:** all images from every subfolder under
`/kaggle/input/datasets/dhamur/cotton-plant-disease` are pooled together
(train/val/test splits inside the original dataset are merged — the leaf
folder name is used as the class label), then re-split for this experiment.

> **Kaggle setup:** Settings → Accelerator → **GPU T4 x2** (P100 has caused
> CUDA capability mismatches with current PyTorch wheels on Kaggle).
> No `pip install` cells are used here to avoid breaking the pre-installed
> RAPIDS/cuDF stack.

Metrics (accuracy / precision / recall / F1, macro-averaged) are saved to
JSON after every round, and the notebook can be re-run from where it left
off (resume support) if the Kaggle session restarts.

In [ ]:
import os
import json
import time
import random
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# -------------------- Config --------------------
DATA_ROOT = "/kaggle/input/datasets/dhamur/cotton-plant-disease"

NUM_CLIENTS   = 5
ALPHA         = 0.5      # Dirichlet concentration parameter

LOCAL_LR      = 0.001
MOMENTUM      = 0.9
BATCH_SIZE    = 32
LOCAL_EPOCHS  = 5
NUM_ROUNDS    = 50
SERVER_LR     = 0.001    # FedOpt server-side Adam learning rate

TEST_SIZE     = 0.2      # held-out, centralized test split for evaluation

WORK_DIR        = "/kaggle/working"
METRICS_PATH    = os.path.join(WORK_DIR, "fedopt_cotton_metrics.json")
CHECKPOINT_PATH = os.path.join(WORK_DIR, "fedopt_cotton_checkpoint.pt")
BEST_MODEL_PATH = os.path.join(WORK_DIR, "fedopt_cotton_best_model.pt")
LABELS_PATH     = os.path.join(WORK_DIR, "fedopt_cotton_label_classes.json")

os.makedirs(WORK_DIR, exist_ok=True)

In [ ]:
# -------------------- Scan ALL subfolders for images --------------------
# Every image anywhere under DATA_ROOT is collected. The class label is
# taken from the immediate parent folder, so train/<class>, val/<class>,
# test/<class> (or any other split layout) all get merged into one class.

VALID_EXT = (".jpg", ".jpeg", ".png")

def scan_dataset(root):
    filepaths, raw_labels = [], []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower().endswith(VALID_EXT):
                filepaths.append(os.path.join(dirpath, fname))
                raw_labels.append(os.path.basename(dirpath))
    return filepaths, raw_labels

filepaths, raw_labels = scan_dataset(DATA_ROOT)
assert len(filepaths) > 0, f"No images found under {DATA_ROOT} — check the dataset path."
print(f"Total images found across all subfolders: {len(filepaths)}")

le = LabelEncoder()
labels = le.fit_transform(raw_labels)
num_classes = len(le.classes_)
print(f"Classes found ({num_classes}):", list(le.classes_))

with open(LABELS_PATH, "w") as f:
    json.dump(list(le.classes_), f, indent=2)

# quick per-class counts
unique, counts = np.unique(labels, return_counts=True)
for cls_idx, cnt in zip(unique, counts):
    print(f"  {le.classes_[cls_idx]:<30s} {cnt}")

In [ ]:
# -------------------- Centralized train/test split --------------------
train_idx, test_idx = train_test_split(
    np.arange(len(filepaths)),
    test_size=TEST_SIZE,
    stratify=labels,
    random_state=SEED,
)

train_paths  = [filepaths[i] for i in train_idx]
train_labels = labels[train_idx]
test_paths   = [filepaths[i] for i in test_idx]
test_labels  = labels[test_idx]

print(f"Train pool: {len(train_paths)} images | Held-out test: {len(test_paths)} images")

# -------------------- Dirichlet non-IID partition across clients --------------------
def dirichlet_partition(labels_arr, num_clients, alpha, num_classes, seed=SEED):
    rng = np.random.default_rng(seed)
    client_indices = [[] for _ in range(num_clients)]
    for c in range(num_classes):
        idx_c = np.where(labels_arr == c)[0]
        rng.shuffle(idx_c)
        proportions = rng.dirichlet(alpha * np.ones(num_clients))
        cut_points = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
        splits = np.split(idx_c, cut_points)
        for i, split in enumerate(splits):
            client_indices[i].extend(split.tolist())
    for i in range(num_clients):
        rng.shuffle(client_indices[i])
    return client_indices

client_indices = dirichlet_partition(train_labels, NUM_CLIENTS, ALPHA, num_classes, seed=SEED)
for i, idxs in enumerate(client_indices):
    print(f"Client {i}: {len(idxs)} samples")

In [ ]:
# -------------------- Dataset / Transforms --------------------
class CottonDataset(Dataset):
    def __init__(self, paths, labs, transform=None):
        self.paths = paths
        self.labels = labs
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
        except Exception:
            # fall back to a blank image rather than crashing a long training run
            img = Image.new("RGB", (224, 224), (0, 0, 0))
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx])

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

client_datasets = []
for idxs in client_indices:
    c_paths = [train_paths[i] for i in idxs]
    c_labels = [train_labels[i] for i in idxs]
    client_datasets.append(CottonDataset(c_paths, c_labels, transform=train_transform))

test_dataset = CottonDataset(test_paths, test_labels, transform=eval_transform)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=(device.type == "cuda"),
)

In [ ]:
# -------------------- Model --------------------
def create_model(n_classes):
    model = models.resnet18(weights=None)  # trained from scratch, no internet required
    model.fc = nn.Linear(model.fc.in_features, n_classes)
    return model

In [ ]:
# -------------------- Local training / aggregation / evaluation --------------------
def local_train(model, dataset, epochs, lr, momentum, batch_size, device):
    model.train()
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=(device.type == "cuda"),
    )
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    criterion = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for imgs, labs in loader:
            imgs, labs = imgs.to(device, non_blocking=True), labs.to(device, non_blocking=True)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labs)
            loss.backward()
            optimizer.step()
    return {k: v.detach().clone().to(device) for k, v in model.state_dict().items()}


def aggregate_states(state_dicts, weights, device):
    total = float(sum(weights))
    keys = state_dicts[0].keys()
    avg_state = {}
    for key in keys:
        stacked = torch.stack(
            [sd[key].to(device).float() * (w / total) for sd, w in zip(state_dicts, weights)],
            dim=0,
        )
        summed = stacked.sum(dim=0)
        avg_state[key] = summed.to(state_dicts[0][key].dtype)
    return avg_state


def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labs in loader:
            imgs = imgs.to(device, non_blocking=True)
            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(labs.numpy().tolist())
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    rec = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}, all_labels, all_preds

In [ ]:
# -------------------- Resume support --------------------
if os.path.exists(METRICS_PATH):
    with open(METRICS_PATH, "r") as f:
        history = json.load(f)
    start_round = len(history)
    print(f"Found existing metrics file — resuming from round {start_round + 1}/{NUM_ROUNDS}")
else:
    history = []
    start_round = 0
    print("No existing metrics file — starting fresh.")

global_model = create_model(num_classes).to(device)
server_optimizer = optim.Adam(global_model.parameters(), lr=SERVER_LR)

if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    global_model.load_state_dict(ckpt["model_state"])
    server_optimizer.load_state_dict(ckpt["optimizer_state"])
    print("Loaded checkpoint — global model and server optimizer state restored.")

best_f1 = max([h["f1"] for h in history], default=-1.0)

In [ ]:
# -------------------- FedOpt training loop --------------------
# Server update (Reddi et al., "Adaptive Federated Optimization"):
#   1. distribute the global model to every client
#   2. each client trains locally with SGD
#   3. server forms the weighted-average client update  avg_state
#   4. pseudo_grad = global_state - avg_state   (this plays the role of a gradient)
#   5. server applies one Adam step using pseudo_grad as the gradient
# Non-trainable buffers (BatchNorm running stats) are just weight-averaged,
# the same way FedAvg would aggregate them — Adam only acts on the
# trainable weights/biases.

param_names = {name for name, _ in global_model.named_parameters()}

for rnd in range(start_round, NUM_ROUNDS):
    t0 = time.time()

    global_state = {k: v.detach().clone() for k, v in global_model.state_dict().items()}

    client_states, client_sizes = [], []
    for cid, dataset in enumerate(client_datasets):
        local_model = create_model(num_classes).to(device)
        local_model.load_state_dict(copy.deepcopy(global_state))
        state = local_train(local_model, dataset, LOCAL_EPOCHS, LOCAL_LR, MOMENTUM, BATCH_SIZE, device)
        client_states.append(state)
        client_sizes.append(len(dataset))
        del local_model
        if device.type == "cuda":
            torch.cuda.empty_cache()

    avg_state = aggregate_states(client_states, client_sizes, device)

    # --- server-side Adam step on trainable parameters ---
    server_optimizer.zero_grad()
    for name, param in global_model.named_parameters():
        pseudo_grad = (global_state[name].to(device) - avg_state[name].to(device)).float()
        param.grad = pseudo_grad.detach().clone()
    server_optimizer.step()

    # --- plain weighted average for non-trainable buffers ---
    updated_state = global_model.state_dict()
    for key in updated_state.keys():
        if key not in param_names:
            updated_state[key] = avg_state[key].to(device)
    global_model.load_state_dict(updated_state)

    metrics, _, _ = evaluate(global_model, test_loader, device)
    metrics["round"] = rnd + 1
    metrics["time_seconds"] = round(time.time() - t0, 2)
    history.append(metrics)

    with open(METRICS_PATH, "w") as f:
        json.dump(history, f, indent=2)

    torch.save(
        {"model_state": global_model.state_dict(), "optimizer_state": server_optimizer.state_dict()},
        CHECKPOINT_PATH,
    )

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        torch.save(global_model.state_dict(), BEST_MODEL_PATH)
        print(f"  -> new best model saved (F1={best_f1:.4f})")

    print(
        f"Round {rnd + 1}/{NUM_ROUNDS} | "
        f"Acc={metrics['accuracy']:.4f} Prec={metrics['precision']:.4f} "
        f"Rec={metrics['recall']:.4f} F1={metrics['f1']:.4f} "
        f"Time={metrics['time_seconds']}s"
    )

print("\nFedOpt training complete.")
print("Best F1 achieved:", round(best_f1, 4))

In [ ]:
# -------------------- Final summary --------------------
print("Per-round metrics:")
for h in history:
    print(
        f"  Round {h['round']:2d} | Acc={h['accuracy']:.4f}  Prec={h['precision']:.4f}  "
        f"Rec={h['recall']:.4f}  F1={h['f1']:.4f}"
    )

# load the best-performing checkpoint and report a full classification report
best_model = create_model(num_classes).to(device)
best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))

best_metrics, all_labels, all_preds = evaluate(best_model, test_loader, device)
print("\nBest model held-out test performance:")
print(best_metrics)

with open(LABELS_PATH, "r") as f:
    class_names = json.load(f)

print("\nClassification report (best model):")
print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))